In [42]:
import pandas as pd


arquivos = [
    'dados_china/china_2000.csv', 
    'dados_china/china_2010.csv', 
    'dados_china/china_2020.csv'
]

lista_dataframes = []

for caminho in arquivos:
    df_temp = pd.read_csv(caminho)
    lista_dataframes.append(df_temp)

df_total = pd.concat(lista_dataframes, ignore_index=True)

df_total.head()
df_total.tail()


C:\Rtemp\ipykernel_20380\2678506274.py:13: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(caminho)
C:\Rtemp\ipykernel_20380\2678506274.py:13: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(caminho)
C:\Rtemp\ipykernel_20380\2678506274.py:13: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(caminho)


,country_id,country_iso3_code,partner_country_id,partner_iso3_code,product_id,product_hs92_code,year,export_value,import_value
10796645,156,CHN,894,ZMB,10030,970110,2024,60867,0
10796646,156,CHN,894,ZMB,10031,970190,2024,28359,0
10796647,156,CHN,894,ZMB,10036,970600,2024,4242,0
10796648,156,CHN,894,ZMB,10037,999999,2024,1411060,12671
10796649,156,CHN,894,ZMB,10039,XXXXXX,2024,0,1637972786


Evolução do comércio da China com os países do G20

In [43]:
#separação dos países do g20

g20_iso = [
    'ARG', 'AUS', 'BRA', 'CAN', 'FRA', 'DEU', 'IND', 'IDN', 'ITA', 'JPN', 
    'KOR', 'MEX', 'RUS', 'SAU', 'ZAF', 'TUR', 'GBR', 'USA', 'EUU'
]

df_g20 = df_total[df_total['partner_iso3_code'].isin(g20_iso)].copy()


#agrupamento por ano e país
evolucao_g20 = df_g20.groupby(['year', 'partner_iso3_code'])[['export_value', 'import_value']].sum().reset_index()

#balança comercial
evolucao_g20['balanca'] = evolucao_g20['export_value'] - evolucao_g20['import_value']


print("dados_g20")

evolucao_g20.head()

dados_g20


,year,partner_iso3_code,export_value,import_value,balanca
0,2000,ARG,775768369,770368683,5399686
1,2000,AUS,4008422283,3479242636,529179647
2,2000,BRA,1096103281,1210626814,-114523533
3,2000,CAN,5729445971,2796130050,2933315921
4,2000,DEU,13710068555,8485381434,5224687121


In [44]:
#nomes 
iso3_para_nome = {
    'ARG': 'Argentina', 'AUS': 'Australia', 'BRA': 'Brazil', 'CAN': 'Canada', 
    'FRA': 'France', 'DEU': 'Germany', 'IND': 'India', 'IDN': 'Indonesia', 
    'ITA': 'Italy', 'JPN': 'Japan', 'KOR': 'South Korea', 'MEX': 'Mexico', 
    'RUS': 'Russia', 'SAU': 'Saudi Arabia', 'ZAF': 'South Africa', 
    'TUR': 'Türkiye', 'GBR': 'United Kingdom', 'USA': 'USA', 'EUU': 'European Union'
}

evolucao_g20['Pais'] = evolucao_g20['partner_iso3_code'].map(iso3_para_nome)

Evolução das exportações da China para países do G20 entre 2000 e 2024

In [51]:
#exportações china p/ g20 (maiores parceiros)

import plotly.graph_objs as go
import plotly.express as px

#separação maiores parceiros
grupo_principais_iso = ['USA', 'JPN', 'DEU', 'AUS', 'KOR', 'BRA', 'RUS']


#visualização
cores_principais = px.colors.qualitative.Dark24

ultimos_valores = []
for iso in grupo_principais_iso:
    df_pais = evolucao_g20[evolucao_g20['partner_iso3_code'] == iso]
    if not df_pais.empty:
        df_pais = df_pais.sort_values('year')
        val = df_pais['export_value'].iloc[-1]
        nome = df_pais['Pais'].iloc[0]
        ultimos_valores.append({'iso': iso, 'val_real': val, 'val_ajustado': val, 'nome': nome})

ultimos_valores.sort(key=lambda x: x['val_real'])

distancia_minima = 12.0 * 10**9 

for _ in range(300):
    for i in range(len(ultimos_valores) - 1):
        if (ultimos_valores[i+1]['val_ajustado'] - ultimos_valores[i]['val_ajustado']) < distancia_minima:
            sobreposicao = distancia_minima - (ultimos_valores[i+1]['val_ajustado'] - ultimos_valores[i]['val_ajustado'])
            ultimos_valores[i]['val_ajustado'] -= sobreposicao / 2
            ultimos_valores[i+1]['val_ajustado'] += sobreposicao / 2
            
posicoes_ajustadas = {item['iso']: item['val_ajustado'] for item in ultimos_valores}

fig1 = go.Figure()

for i, iso in enumerate(grupo_principais_iso):
    df_pais = evolucao_g20[evolucao_g20['partner_iso3_code'] == iso].sort_values('year')
    
    if not df_pais.empty:
        nome_pais = df_pais['Pais'].iloc[0]
        cor = cores_principais[i % len(cores_principais)]
        
        
        fig1.add_trace(
            go.Scatter(
                x=df_pais['year'], 
                y=df_pais['export_value'], 
                name=nome_pais, 
                mode='lines+markers', 
                line=dict(color=cor, width=2.5),
                showlegend=False))
        
    
        fig1.add_annotation(
            x=df_pais['year'].iloc[-1], 
            y=posicoes_ajustadas[iso],
            text=nome_pais, 
            showarrow=False, 
            xanchor='left', 
            yanchor='middle',
            xshift=8, 
            font=dict(color=cor, size=10))


fig1.update_layout(
    title=dict(text="Evolution of China's Exports: Major G20 Partners", x=0.5),
    template="plotly_white",
    height=550, 
    hovermode="x unified",
    margin=dict(r=140, l=60, t=60, b=60))

fig1.update_yaxes(title_text="USD Value")
fig1.update_xaxes(title_text="Year", range=[1999, 2026.5], rangeslider_visible=False)

fig1.show()

In [52]:
#Exportação china para g20 (demais países)

import plotly.graph_objs as go
import plotly.express as px

#separação demais países
grupo_restante_iso = ['CAN', 'FRA', 'ITA', 'GBR', 'ARG', 'IND', 'IDN', 'MEX', 'SAU', 'ZAF', 'TUR', 'EUU']


#visualização
cores_restante = px.colors.qualitative.Dark24

ultimos_valores = []
for iso in grupo_restante_iso:
    df_pais = evolucao_g20[evolucao_g20['partner_iso3_code'] == iso]
    if not df_pais.empty:
        df_pais = df_pais.sort_values('year')
        val = df_pais['export_value'].iloc[-1]
        nome = df_pais['Pais'].iloc[0]
        ultimos_valores.append({'iso': iso, 'val_real': val, 'val_ajustado': val, 'nome': nome})

ultimos_valores.sort(key=lambda x: x['val_real'])


distancia_minima = 3.5 * 10**9 

for _ in range(300):
    for i in range(len(ultimos_valores) - 1):
        if (ultimos_valores[i+1]['val_ajustado'] - ultimos_valores[i]['val_ajustado']) < distancia_minima:
            sobreposicao = distancia_minima - (ultimos_valores[i+1]['val_ajustado'] - ultimos_valores[i]['val_ajustado'])
            ultimos_valores[i]['val_ajustado'] -= sobreposicao / 2
            ultimos_valores[i+1]['val_ajustado'] += sobreposicao / 2
            
posicoes_ajustadas = {item['iso']: item['val_ajustado'] for item in ultimos_valores}


fig2 = go.Figure()

for i, iso in enumerate(grupo_restante_iso):
    df_pais = evolucao_g20[evolucao_g20['partner_iso3_code'] == iso].sort_values('year')
    
    if not df_pais.empty:
        nome_pais = df_pais['Pais'].iloc[0]
        cor = cores_restante[i % len(cores_restante)]
        
    
        fig2.add_trace(
            go.Scatter(
                x=df_pais['year'], 
                y=df_pais['export_value'],
                name=nome_pais, 
                mode='lines+markers', 
                line=dict(color=cor, width=2.5),
                showlegend=False))
        
        
        fig2.add_annotation(
            x=df_pais['year'].iloc[-1], 
            y=posicoes_ajustadas[iso],
            text=nome_pais, 
            showarrow=False, 
            xanchor='left', 
            yanchor='middle',
            xshift=8, 
            font=dict(color=cor, size=10))


fig2.update_layout(
    title=dict(text="Evolution of China's Exports: Other G20 Partners", x=0.5),
    template="plotly_white",
    height=550, 
    hovermode="x unified",
    margin=dict(r=140, l=60, t=60, b=60))

fig2.update_yaxes(title_text="USD Value")
fig2.update_xaxes(title_text="Year", range=[1999, 2026.5], rangeslider_visible=False)

fig2.show()

Evolução das exportações dos países do G20 para a China entre 2000 e 2024

In [53]:
#exportações g20 para china (maiores parceiros)

import plotly.graph_objs as go
import plotly.express as px

#separação maiores parceiros
grupo_principais_iso = ['USA', 'JPN', 'DEU', 'AUS', 'KOR', 'BRA', 'RUS']

#visualização
cores_principais = px.colors.qualitative.Dark24

ultimos_valores = []
for iso in grupo_principais_iso:
    df_pais = evolucao_g20[evolucao_g20['partner_iso3_code'] == iso]
    if not df_pais.empty:
        df_pais = df_pais.sort_values('year')
        val = df_pais['import_value'].iloc[-1]
        nome = df_pais['Pais'].iloc[0]
        ultimos_valores.append({'iso': iso, 'val_real': val, 'val_ajustado': val, 'nome': nome})

ultimos_valores.sort(key=lambda x: x['val_real'])

distancia_minima = 5.0 * 10**9 

for _ in range(300):
    for i in range(len(ultimos_valores) - 1):
        if (ultimos_valores[i+1]['val_ajustado'] - ultimos_valores[i]['val_ajustado']) < distancia_minima:
            sobreposicao = distancia_minima - (ultimos_valores[i+1]['val_ajustado'] - ultimos_valores[i]['val_ajustado'])
            ultimos_valores[i]['val_ajustado'] -= sobreposicao / 2
            ultimos_valores[i+1]['val_ajustado'] += sobreposicao / 2
            
posicoes_ajustadas = {item['iso']: item['val_ajustado'] for item in ultimos_valores}

fig1 = go.Figure()

for i, iso in enumerate(grupo_principais_iso):
    df_pais = evolucao_g20[evolucao_g20['partner_iso3_code'] == iso].sort_values('year')
    
    if not df_pais.empty:
        nome_pais = df_pais['Pais'].iloc[0]
        cor = cores_principais[i % len(cores_principais)]
        
    
        fig1.add_trace(
            go.Scatter(
                x=df_pais['year'], 
                y=df_pais['import_value'], 
                name=nome_pais, 
                mode='lines+markers', 
                line=dict(color=cor, width=2.5),
                showlegend=False))
        
    
        fig1.add_annotation(
            x=df_pais['year'].iloc[-1], 
            y=posicoes_ajustadas[iso],
            text=nome_pais, 
            showarrow=False, 
            xanchor='left', 
            yanchor='middle',
            xshift=8, 
            font=dict(color=cor, size=10))

fig1.update_layout(
    title=dict(text="Evolution of Exports to China: Major G20 Partners", x=0.5),
    template="plotly_white",
    height=550, 
    hovermode="x unified",
    margin=dict(r=140, l=60, t=60, b=60))

fig1.update_yaxes(title_text="USD Value")
fig1.update_xaxes(title_text="Year", range=[1999, 2026.5], rangeslider_visible=False)

fig1.show()

In [54]:
##exportações g20 para china (demais países)

import plotly.graph_objs as go
import plotly.express as px

#separação dos parceiros
grupo_restante_iso = ['CAN', 'FRA', 'ITA', 'GBR', 'ARG', 'IND', 'IDN', 'MEX', 'SAU', 'ZAF', 'TUR', 'EUU']


cores_restante = px.colors.qualitative.Dark24

ultimos_valores = []
for iso in grupo_restante_iso:
    df_pais = evolucao_g20[evolucao_g20['partner_iso3_code'] == iso]
    if not df_pais.empty:
        df_pais = df_pais.sort_values('year')
        val = df_pais['import_value'].iloc[-1]
        nome = df_pais['Pais'].iloc[0]
        ultimos_valores.append({'iso': iso, 'val_real': val, 'val_ajustado': val, 'nome': nome})

ultimos_valores.sort(key=lambda x: x['val_real'])

distancia_minima = 2.2 * 10**9 

for _ in range(300):
    for i in range(len(ultimos_valores) - 1):
        if (ultimos_valores[i+1]['val_ajustado'] - ultimos_valores[i]['val_ajustado']) < distancia_minima:
            sobreposicao = distancia_minima - (ultimos_valores[i+1]['val_ajustado'] - ultimos_valores[i]['val_ajustado'])
            ultimos_valores[i]['val_ajustado'] -= sobreposicao / 2
            ultimos_valores[i+1]['val_ajustado'] += sobreposicao / 2
            
posicoes_ajustadas = {item['iso']: item['val_ajustado'] for item in ultimos_valores}


fig2 = go.Figure()

for i, iso in enumerate(grupo_restante_iso):
    df_pais = evolucao_g20[evolucao_g20['partner_iso3_code'] == iso].sort_values('year')
    
    if not df_pais.empty:
        nome_pais = df_pais['Pais'].iloc[0]
        cor = cores_restante[i % len(cores_restante)]
        
    
        fig2.add_trace(
            go.Scatter(
                x=df_pais['year'], 
                y=df_pais['import_value'], 
                name=nome_pais, 
                mode='lines+markers', 
                line=dict(color=cor, width=2.5),
                showlegend=False))
        
        
        fig2.add_annotation(
            x=df_pais['year'].iloc[-1], 
            y=posicoes_ajustadas[iso],
            text=nome_pais, 
            showarrow=False, 
            xanchor='left', 
            yanchor='middle',
            xshift=8, 
            font=dict(color=cor, size=10))


fig2.update_layout(
    title=dict(text="Evolution of Exports to China: Other G20 Partners", x=0.5),
    template="plotly_white",
    height=550, 
    hovermode="x unified",
    margin=dict(r=140, l=60, t=60, b=60))

fig2.update_yaxes(title_text="USD Value")
fig2.update_xaxes(title_text="Year", range=[1999, 2026.5], rangeslider_visible=False)

fig2.show()